# IDO1 Molecular Docking Pipeline
**Target:** IDO1 (PDB: 9FOZ) | **Analyst:** Shipra | **Tool:** AutoDock Vina 1.2.7

Colab-compatible notebook reproducing S6–S10 of the IDO1 docking pipeline.

In [ ]:
# Install dependencies
!pip install rdkit meeko gemmi -q

In [ ]:
# Colab: mount Drive and set working directory
# from google.colab import drive
# drive.mount('/content/drive')
import os
# os.chdir('/content/drive/MyDrive/AIMS_capstone')  # adjust as needed
print('Working dir:', os.getcwd())

In [ ]:
# S6 — Ligand Preparation
import pandas as pd, os, subprocess
from rdkit import Chem
from rdkit.Chem import AllChem

HANDOVER = 'tryptanthrin-anticancer-drug-discovery/05_ML_Pipeline_Validation_Integration/reports/diversity_shortlist/master_docking_handover.csv'
df = pd.read_csv(HANDOVER)
ido1 = df[df['Target']=='IDO1'][['Analog_ID','SMILES']].reset_index(drop=True)
print(f'IDO1 analogs: {len(ido1)}')

os.makedirs('IDO1/04_ligand_prep', exist_ok=True)

results = []
for _, row in ido1.iterrows():
    aid, smi = row['Analog_ID'], row['SMILES']
    sdf   = f'IDO1/04_ligand_prep/{aid}.sdf'
    pdbqt = f'IDO1/04_ligand_prep/{aid}.pdbqt'
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    w = Chem.SDWriter(sdf); w.write(mol); w.close()
    r = subprocess.run(['mk_prepare_ligand', '-i', sdf, '-o', pdbqt],
                       capture_output=True, text=True)
    status = 'OK' if os.path.exists(pdbqt) else 'FAIL'
    results.append({'Analog_ID': aid, 'Status': status})
    print(f'  {aid}: {status}')
ok = sum(1 for r in results if r['Status']=='OK')
print(f'Passed: {ok}/10')

In [ ]:
# S7 — Vina Docking
import re, glob

os.makedirs('IDO1/05_docking', exist_ok=True)

# On Colab: !wget https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64
# !chmod +x vina_1.2.7_linux_x86_64; !mv vina_1.2.7_linux_x86_64 vina
VINA     = './vina'  # adjust path
RECEPTOR = 'IDO1/00_receptor/IDO1.pdbqt'

scores = []
for aid in ido1['Analog_ID']:
    ligand  = f'IDO1/04_ligand_prep/{aid}.pdbqt'
    out     = f'IDO1/05_docking/{aid}_docked.pdbqt'
    logfile = f'IDO1/05_docking/{aid}.log'
    cmd = [VINA,
           '--receptor', RECEPTOR, '--ligand', ligand,
           '--center_x', '-10.985', '--center_y', '29.055', '--center_z', '-35.034',
           '--size_x', '22', '--size_y', '22', '--size_z', '22',
           '--exhaustiveness', '32', '--num_modes', '9', '--seed', '42',
           '--out', out]
    r = subprocess.run(cmd, capture_output=True, text=True)
    with open(logfile, 'w') as f: f.write(r.stdout + r.stderr)
    best = None
    for line in (r.stdout+r.stderr).splitlines():
        m = re.match(r'\s+1\s+([-\d.]+)', line)
        if m: best = float(m.group(1)); break
    scores.append({'Analog_ID': aid, 'Vina_best_kcal_mol': best})
    print(f'{aid}: {best}')

score_df = pd.DataFrame(scores).sort_values('Vina_best_kcal_mol')
score_df.to_csv('IDO1/05_docking/IDO1_vina_scores.csv', index=False)
print(score_df.to_string(index=False))

In [ ]:
# S8 — Interaction Analysis (Top-3 analogs)
import numpy as np

KEY_POINTS = {
    'Fe_HEM':     np.array([-14.947, 32.621, -32.297]),
    'Arg231_CZ':  np.array([-8.5,   28.3,  -31.1]),
    'His346_NE2': np.array([-14.2,  33.1,  -28.6]),
    'Ser167_OG':  np.array([-10.1,  24.7,  -34.8]),
    'Phe163_CZ':  np.array([-7.3,   27.0,  -37.2]),
}

def parse_pdbqt_model1(path):
    atoms, in_m1 = [], False
    with open(path) as f:
        for line in f:
            if line.startswith('MODEL'): in_m1 = int(line.split()[1])==1; continue
            if line.startswith('ENDMDL') and in_m1: break
            if in_m1 and line.startswith(('ATOM','HETATM')):
                aname = line[12:16].strip()
                elem  = line[76:78].strip() if len(line)>76 else aname[0]
                x,y,z = float(line[30:38]),float(line[38:46]),float(line[46:54])
                atoms.append((aname,elem,x,y,z))
    return atoms

top3 = score_df.head(3)['Analog_ID'].tolist()
os.makedirs('IDO1/06_interactions', exist_ok=True)
hbond_rows, hydro_rows = [], []

for aid in top3:
    atoms  = parse_pdbqt_model1(f'IDO1/05_docking/{aid}_docked.pdbqt')
    coords = np.array([[a[2],a[3],a[4]] for a in atoms])
    elems  = [a[1] for a in atoms]
    for site, ref in KEY_POINTS.items():
        dists = np.linalg.norm(coords - ref, axis=1)
        mi, md = int(np.argmin(dists)), float(np.min(dists))
        elem = elems[mi]
        if site=='Fe_HEM' and md<=3.0:
            hbond_rows.append({'Analog_ID':aid,'Interaction':'Fe_coordination','Site':site,'Ligand_atom':atoms[mi][0],'Element':elem,'Distance_A':round(md,3)})
        elif elem in ('N','O') and md<=3.5:
            hbond_rows.append({'Analog_ID':aid,'Interaction':'H-bond','Site':site,'Ligand_atom':atoms[mi][0],'Element':elem,'Distance_A':round(md,3)})
        elif elem=='C' and md<=4.0:
            hydro_rows.append({'Analog_ID':aid,'Interaction':'Hydrophobic','Site':site,'Ligand_atom':atoms[mi][0],'Element':elem,'Distance_A':round(md,3)})

hb_df = pd.DataFrame(hbond_rows)
hy_df = pd.DataFrame(hydro_rows)
hb_df.to_csv('IDO1/06_interactions/IDO1_hbond_geometry.csv', index=False)
hy_df.to_csv('IDO1/06_interactions/IDO1_hydrophobic_contacts.csv', index=False)
print('H-bonds/Coordination:'); print(hb_df.to_string(index=False))
print('Hydrophobic:');           print(hy_df.to_string(index=False))

In [ ]:
# S9 — Integrated Ranking
ml      = pd.read_csv(HANDOVER)
ido1_ml = ml[ml['Target']=='IDO1'].copy()
merged  = ido1_ml.merge(score_df, on='Analog_ID', how='left')

vc = 'Vina_best_kcal_mol'
pc = 'Consensus_pIC50'
qc = 'QED'
merged['Vina_score']  = 1-(merged[vc]-merged[vc].min())/(merged[vc].max()-merged[vc].min()+1e-9)
merged['pIC50_norm']  = (merged[pc]-merged[pc].min())/(merged[pc].max()-merged[pc].min()+1e-9)
merged['QED_norm']    = (merged[qc]-merged[qc].min())/(merged[qc].max()-merged[qc].min()+1e-9)
pains = merged.get('PAINS_flag', pd.Series(False, index=merged.index)).astype(float)
brenk = merged.get('Brenk_flag', pd.Series(False, index=merged.index)).astype(float)
merged['Composite_score'] = (0.40*merged['Vina_score'] + 0.40*merged['pIC50_norm']
                            + 0.20*merged['QED_norm'] - 0.10*pains - 0.10*brenk)
merged['Rank'] = merged['Composite_score'].rank(ascending=False).astype(int)
merged.sort_values('Rank', inplace=True)

os.makedirs('IDO1/07_ranking', exist_ok=True)
merged.to_csv('IDO1/07_ranking/IDO1_final_candidate_ranking.csv', index=False)
show = ['Rank','Analog_ID','Vina_best_kcal_mol','Consensus_pIC50','QED','PAINS_flag','Brenk_flag','Composite_score']
show = [c for c in show if c in merged.columns]
print(merged[show].to_string(index=False))
print(f"\nTop candidate: {merged.iloc[0]['Analog_ID']} | Vina={merged.iloc[0]['Vina_best_kcal_mol']:.3f} | Composite={merged.iloc[0]['Composite_score']:.3f}")

## Results Summary

| Rank | Analog_ID | Vina (kcal/mol) | Consensus pIC50 | QED | PAINS | Composite |
|------|-----------|-----------------|-----------------|-----|-------|----------|
| **1** | TRY_b79da49d45504836 | −10.91 | 6.553 | 0.563 | ✗ | **0.867** |
| 2 | TRY_62b0878d0fc0ffb9 | −10.26 | 6.401 | 0.576 | ✗ | 0.560 |
| 3 | TRY_610c82dc1e8f7d06 | −9.496 | 6.629 | 0.502 | ✗ | 0.538 |

**Primary candidate: TRY_b79da49d45504836**  
- Best Vina: −10.91 kcal/mol  
- H-bond with Ser167_OG (1.21 Å)  
- Piperazine substituent — enhanced solubility  
- No PAINS/Brenk flags  

> All results are computational predictions. Experimental validation required.